# NLP Weekly Task 6: GloVe Word Embedding

**GloVe (Global Vectors for Word Representation)** is an unsupervised learning algorithm that generates dense word embeddings by analyzing co-occurrence patterns in a large text corpus, capturing semantic relationships between words.

- Uses a word co-occurrence matrix to learn relationships between words
- Combines global statistical information (LSA) with local context-based learning (like Word2Vec)
- Optimizes embeddings so the dot product approximates Pointwise Mutual Information (PMI)
- Captures both semantic and syntactic relationships

**Requirements:**
```
pip install tensorflow numpy gensim
```


## Part 1: How GloVe Works (Step by Step)

### 1. Preprocess the Text

First, split the text into individual words (tokenization).

In [1]:
input_text = "The peon is ringing the bell"
tokenized_words = input_text.split()
print("Input text:", input_text)
print("Tokenized words:", tokenized_words)

Input text: The peon is ringing the bell
Tokenized words: ['The', 'peon', 'is', 'ringing', 'the', 'bell']


### 2. Creating the Vocabulary

Create a list of all unique words in the text and count how often each word appears.

In [2]:
from collections import Counter

word_freq = Counter(tokenized_words)
print("Vocabulary with word frequencies:")
print(dict(word_freq))

Vocabulary with word frequencies:
{'The': 1, 'peon': 1, 'is': 1, 'ringing': 1, 'the': 1, 'bell': 1}


### 3. Building a Co-occurrence Matrix

We count how often each word appears near other words within a fixed context window.
Here we use a window size of 2 (2 words before and after each word).

In [3]:
import numpy as np
import pandas as pd

def build_cooccurrence_matrix(tokens, window_size=2):
    vocab = sorted(set(tokens))
    idx = {w: i for i, w in enumerate(vocab)}
    matrix = np.zeros((len(vocab), len(vocab)), dtype=int)

    for center_i, center_word in enumerate(tokens):
        start = max(0, center_i - window_size)
        end = min(len(tokens), center_i + window_size + 1)
        for context_i in range(start, end):
            if context_i == center_i:
                continue
            matrix[idx[center_word], idx[tokens[context_i]]] += 1

    return pd.DataFrame(matrix, index=vocab, columns=vocab)

cooc_matrix = build_cooccurrence_matrix(tokenized_words, window_size=2)
cooc_matrix

,The,bell,is,peon,ringing,the
The,0,0,1,1,0,0
bell,0,0,0,0,1,1
is,1,0,0,1,1,1
peon,1,0,1,0,1,0
ringing,0,1,1,1,0,1
the,0,1,1,0,1,0


> The value at (i, j) represents how often word *i* and word *j* appear together in the context window.

### 4 & 5. Dot Product & Training

GloVe learns word vectors such that the **dot product of two word vectors reflects how often the words co-occur**. Words that appear in similar contexts (like "The" and "is") end up with similar vector representations, while words that rarely co-occur (like "peon" and "bell") end up far apart. Training minimizes the difference between the predicted dot product and the actual co-occurrence statistics (using Pointwise Mutual Information as the target signal).

### 6. Embedding Matrix

After training (on a real corpus, this happens over millions of sentences), each word is represented by a dense vector — the embedding matrix.

## Part 2: Implementation — Building a Vocabulary with Keras Tokenizer

### 1. Importing Libraries

In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

I0000 00:00:1786678338.676052     564 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786678338.676579     564 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1786678338.725053     564 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1786678340.021361     564 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1786678340.021730     564 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


### 2. Creating Vocabulary

Define a list of words representing our small sample corpus.

In [5]:
texts = ['text', 'the', 'leader', 'prime', 'natural', 'language']

### 3. Initializing and Fitting the Tokenizer

`fit_on_texts` processes the corpus and builds a word-to-index mapping.

In [6]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(texts)

print("Number of unique words in dictionary =", len(tokenizer.word_index))
print("Dictionary is =", tokenizer.word_index)

Number of unique words in dictionary = 6
Dictionary is = {'text': 1, 'the': 2, 'leader': 3, 'prime': 4, 'natural': 5, 'language': 6}


## Part 3: Using Real Pretrained GloVe Vectors

Rather than training GloVe from scratch (which needs a huge corpus), we typically **load pretrained vectors**. Here we use `gensim` to download the official `glove-wiki-gigaword-50` vectors (50-dimensional, trained on Wikipedia + Gigaword).

In [7]:
import gensim.downloader as api

# Downloads (and caches) the 50-dimensional GloVe vectors trained on
# Wikipedia 2014 + Gigaword 5 (6B tokens, 400K vocab)
glove_vectors = api.load("glove-wiki-gigaword-50")

print("Vocabulary size:", len(glove_vectors.index_to_key))
print("Vector dimension:", glove_vectors.vector_size)

Vocabulary size: 400000
Vector dimension: 50


### Look up embeddings for our vocabulary words

In [8]:
for word in ['text', 'the', 'leader', 'prime', 'natural', 'language']:
    if word in glove_vectors:
        print(f"'{word}' vector (first 8 dims): {np.round(glove_vectors[word][:8], 3)}")
    else:
        print(f"'{word}' not in vocabulary")

'text' vector (first 8 dims): [ 0.326  0.367 -0.007 -0.376  0.667  0.216 -0.198 -1.1  ]
'the' vector (first 8 dims): [ 0.418  0.25  -0.412  0.122  0.345 -0.044 -0.497 -0.179]
'leader' vector (first 8 dims): [-0.157  0.261  0.789  0.652  1.2    0.354 -0.343  0.317]
'prime' vector (first 8 dims): [ 0.508  0.699  0.415  0.5    0.827  0.589 -0.434  0.217]
'natural' vector (first 8 dims): [ 0.443  0.848 -0.46   0.68   0.138  0.395 -0.173 -0.641]
'language' vector (first 8 dims): [-0.58  -0.11  -1.156 -0.003 -0.206  0.453 -0.167 -1.038]


### Build an embedding matrix for our vocabulary

This is the typical step before feeding embeddings into a Keras `Embedding` layer.

In [9]:
embedding_dim = glove_vectors.vector_size
vocab_size = len(tokenizer.word_index) + 1  # +1 for padding/index 0

embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, i in tokenizer.word_index.items():
    if word in glove_vectors:
        embedding_matrix[i] = glove_vectors[word]

print("Embedding matrix shape:", embedding_matrix.shape)
pd.DataFrame(
    embedding_matrix,
    index=['<pad>'] + list(tokenizer.word_index.keys())
).round(3)

Embedding matrix shape: (7, 50)


,0,1,2,3,4,5,6,7,8,9,...,40,41,42,43,44,45,46,47,48,49
<pad>,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000
text,0.326,0.367,-0.007,-0.376,0.667,0.216,-0.198,-1.100,-0.422,0.106,...,1.630,-0.172,-0.174,-0.010,-0.178,0.931,1.038,0.943,-0.148,-0.611
the,0.418,0.250,-0.412,0.122,0.345,-0.044,-0.497,-0.179,-0.001,-0.657,...,-0.299,-0.157,-0.348,-0.046,-0.443,0.188,0.003,-0.184,-0.115,-0.786
leader,-0.157,0.261,0.789,0.652,1.200,0.354,-0.343,0.317,-1.150,-0.161,...,-0.091,0.750,-1.315,-0.754,0.829,0.051,-1.480,-0.111,0.271,-0.487
prime,0.508,0.699,0.415,0.500,0.827,0.589,-0.434,0.217,-1.818,-0.743,...,-0.329,1.326,-1.180,-1.386,0.202,0.515,-1.907,0.654,1.725,-0.601
natural,0.443,0.848,-0.460,0.680,0.138,0.395,-0.173,-0.641,0.864,0.816,...,-0.214,0.159,0.522,0.206,-0.167,0.581,-0.368,0.036,0.014,-0.248
language,-0.580,-0.110,-1.156,-0.003,-0.206,0.453,-0.167,-1.038,-0.992,0.399,...,0.826,0.571,0.212,0.469,-0.600,0.299,0.679,1.424,-0.032,-0.126


### Exploring semantic relationships

Since GloVe captures semantic meaning, we can find the most similar words and even solve word analogies.

In [10]:
# Most similar words
print("Words most similar to 'language':")
for word, score in glove_vectors.most_similar('language', topn=5):
    print(f"  {word}: {score:.3f}")

Words most similar to 'language':
  languages: 0.881
  word: 0.810
  spoken: 0.807
  vocabulary: 0.790
  translation: 0.788


In [11]:
# Classic word analogy: king - man + woman = queen
result = glove_vectors.most_similar(positive=['king', 'woman'], negative=['man'], topn=3)
print("king - man + woman =")
for word, score in result:
    print(f"  {word}: {score:.3f}")

king - man + woman =
  queen: 0.852
  throne: 0.766
  prince: 0.759


## Summary

- GloVe builds embeddings from a **global word co-occurrence matrix**, unlike Word2Vec which only looks at local context windows.
- The core idea: **dot product of two word vectors ≈ their co-occurrence statistics** (via PMI).
- In practice, we rarely train GloVe from scratch — we load **pretrained vectors** (like `glove-wiki-gigaword-50`) and either use them directly for similarity/analogy tasks, or build an **embedding matrix** to initialize a neural network's `Embedding` layer.
- GloVe vectors capture both **semantic** relationships (e.g. `king - man + woman ≈ queen`) and **syntactic** ones (e.g. verb tense, plurals).